# ЭКЗАМЕНАЦИОННАЯ ЗАДАЧА №03: Выбор ценовой стратегии для AI-сервиса по созданию контента
# Дата: 2026-09-19
# Студент: Орлова Екатерина Александровна
# Группа: ПИ 4-2

## Общая логика решения

**Исходные данные:**
- Рынок: 500 000 пользователей
- Затраты на разработку: 15 млн руб.
- Ежемесячные затраты: 2 млн руб.
- CAC: реклама — 1500 руб., контент-маркетинг — 800 руб., виральный — 100 руб.
- Ставка дисконтирования: 12% годовых
- Горизонт: 3 года
- Начальная база: 1000 пользователей в Год 0

**Стратегии:**

| Стратегия | Цена | Конверсия (платные) | Churn |
|---|---|---|---|
| Freemium | 2000 руб./мес | 2% | 8% |
| Подписка | 1500 руб./мес | 20% (пробный) | 5% |
| Динамическая | 500–2000 руб./мес | 15% | 6% |
| Персонализированная | 300–5000 руб. | 12% | 4% |

> Примечание: для динамической и персонализированной стратегий возьмём среднюю цену: (500+2000)/2 = 1250 руб. и (300+5000)/2 = 2650 руб. соответственно. Это разумное допущение, которое нужно явно указать.


## Часть 1. Расчёт ключевых метрик

### ARPU
Формула: `ARPU = Цена × Конверсия`

### LTV
Формула: `LTV = ARPU / Churn`

### LTV/CAC
Считаем для каждого канала привлечения и каждой стратегии.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Исходные данные ---
strategies = {
    'Freemium': {'price': 2000, 'conversion': 0.02, 'churn': 0.08},
    'Подписка': {'price': 1500, 'conversion': 0.20, 'churn': 0.05},
    'Динамическая': {'price': 1250, 'conversion': 0.15, 'churn': 0.06},  # средняя цена
    'Персонализированная': {'price': 2650, 'conversion': 0.12, 'churn': 0.04},  # средняя цена
}

cac_channels = {
    'Реклама': 1500,
    'Контент-маркетинг': 800,
    'Виральный рост': 100,
}

# --- Расчёт ARPU и LTV ---
rows = []
for name, s in strategies.items():
    arpu = s['price'] * s['conversion']
    ltv = arpu / s['churn']
    rows.append({'Стратегия': name, 'Цена': s['price'], 'Конверсия': s['conversion'],
                 'Churn': s['churn'], 'ARPU': arpu, 'LTV': ltv})

df_metrics = pd.DataFrame(rows)
print("=== ARPU и LTV ===")
print(df_metrics.to_string(index=False))

# --- LTV/CAC для каждого канала ---
ltv_cac_rows = []
for name, s in strategies.items():
    arpu = s['price'] * s['conversion']
    ltv = arpu / s['churn']
    for channel, cac in cac_channels.items():
        ratio = ltv / cac
        ltv_cac_rows.append({'Стратегия': name, 'Канал': channel, 'CAC': cac,
                             'LTV': ltv, 'LTV/CAC': ratio})

df_ltv_cac = pd.DataFrame(ltv_cac_rows)
print("\n=== LTV/CAC по каналам ===")
print(df_ltv_cac.to_string(index=False))

=== ARPU и LTV ===
          Стратегия  Цена  Конверсия  Churn  ARPU    LTV
           Freemium  2000       0.02   0.08  40.0  500.0
           Подписка  1500       0.20   0.05 300.0 6000.0
       Динамическая  1250       0.15   0.06 187.5 3125.0
Персонализированная  2650       0.12   0.04 318.0 7950.0

=== LTV/CAC по каналам ===
          Стратегия             Канал  CAC    LTV   LTV/CAC
           Freemium           Реклама 1500  500.0  0.333333
           Freemium Контент-маркетинг  800  500.0  0.625000
           Freemium    Виральный рост  100  500.0  5.000000
           Подписка           Реклама 1500 6000.0  4.000000
           Подписка Контент-маркетинг  800 6000.0  7.500000
           Подписка    Виральный рост  100 6000.0 60.000000
       Динамическая           Реклама 1500 3125.0  2.083333
       Динамическая Контент-маркетинг  800 3125.0  3.906250
       Динамическая    Виральный рост  100 3125.0 31.250000
Персонализированная           Реклама 1500 7950.0  5.300000
Персонал

**Ожидаемые результаты (пример):**

| Стратегия | ARPU | LTV |
|---|---|---|
| Freemium | 40 | 500 |
| Подписка | 300 | 6000 |
| Динамическая | 187.5 | 3125 |
| Персонализированная | 318 | 7950 |

LTV/CAC для рекламы (CAC=1500):
- Freemium: 0.33 (убыточно)
- Подписка: 4.0 (хорошо)
- Динамическая: 2.08 (приемлемо)
- Персонализированная: 5.3 (отлично)



## Часть 2. Оценка эффективности

### Модель роста пользователей
- Год 0: 1000 пользователей
- Год 1: +100% → 2000
- Год 2: +50% → 3000
- Год 3: +20% → 3600

Но это общая база. Платящие пользователи = база × конверсия.

### Прибыль
```
Выручка = Платящие × Цена × 12
Затраты = Постоянные (2 млн × 12) + CAC × Прирост пользователей
Прибыль = Выручка − Затраты
```

Учтём, что CAC зависит от канала. Для простоты возьмём средневзвешенный CAC или рассмотрим отдельно. В задаче не указано распределение по каналам, поэтому возьмём средний CAC = (1500+800+100)/3 = 800 руб.

### NPV
```
NPV = Σ (Прибыль_t / (1 + r)^t) − Начальные инвестиции
```
где r = 12%, t = 1, 2, 3. Начальные инвестиции = 15 млн руб.


In [5]:
# --- Прогноз прибыли и NPV ---
initial_investment = 15_000_000
monthly_fixed_cost = 2_000_000
discount_rate = 0.12
avg_cac = np.mean(list(cac_channels.values()))  # 800 руб.

growth = {1: 1.0, 2: 0.5, 3: 0.2}  # прирост
base_users = 1000

def project_users(base, years=3):
    users = {0: base}
    for y in range(1, years+1):
        users[y] = users[y-1] * (1 + growth[y])
    return users

users = project_users(base_users)
print("Пользователи по годам:", users)

npv_rows = []
for name, s in strategies.items():
    paying = {y: users[y] * s['conversion'] for y in range(1, 4)}
    revenue = {y: paying[y] * s['price'] * 12 for y in range(1, 4)}

    # прирост пользователей для CAC
    new_users = {y: users[y] - users[y-1] for y in range(1, 4)}
    cac_cost = {y: new_users[y] * avg_cac for y in range(1, 4)}
    fixed_cost = {y: monthly_fixed_cost * 12 for y in range(1, 4)}

    profit = {y: revenue[y] - cac_cost[y] - fixed_cost[y] for y in range(1, 4)}
    npv = sum(profit[y] / (1 + discount_rate)**y for y in range(1, 4)) - initial_investment

    npv_rows.append({'Стратегия': name,
                     'Прибыль Год 1': profit[1],
                     'Прибыль Год 2': profit[2],
                     'Прибыль Год 3': profit[3],
                     'NPV': npv})

df_npv = pd.DataFrame(npv_rows)
print("\n=== Прогноз прибыли и NPV ===")
print(df_npv.to_string(index=False))


Пользователи по годам: {0: 1000, 1: 2000.0, 2: 3000.0, 3: 3600.0}

=== Прогноз прибыли и NPV ===
          Стратегия  Прибыль Год 1  Прибыль Год 2  Прибыль Год 3           NPV
           Freemium    -23840000.0    -23360000.0    -22752000.0 -7.110259e+07
           Подписка    -17600000.0    -14000000.0    -11520000.0 -5.007471e+07
       Динамическая    -20300000.0    -18050000.0    -16380000.0 -5.917331e+07
Персонализированная    -17168000.0    -13352000.0    -10742400.0 -4.861893e+07


**Важно:** При таких параметрах большинство стратегий, скорее всего, покажут отрицательный NPV из-за высоких постоянных затрат (24 млн руб./год) и малой базы пользователей. Это нормально для стартапа — нужно показать, что вы это понимаете, и предложить корректировки (например, снижение затрат или более агрессивный рост).

## Часть 3. Анализ чувствительности

Изменяем конверсию, churn и CAC на ±30% и смотрим, как меняется NPV.


In [6]:
# --- Анализ чувствительности ---
def calculate_npv(strategy_params, cac_value, conversion_mult=1.0, churn_mult=1.0):
    s = strategy_params.copy()
    s['conversion'] *= conversion_mult
    s['churn'] *= churn_mult

    users = project_users(base_users)
    paying = {y: users[y] * s['conversion'] for y in range(1, 4)}
    revenue = {y: paying[y] * s['price'] * 12 for y in range(1, 4)}
    new_users = {y: users[y] - users[y-1] for y in range(1, 4)}
    cac_cost = {y: new_users[y] * cac_value for y in range(1, 4)}
    fixed_cost = {y: monthly_fixed_cost * 12 for y in range(1, 4)}
    profit = {y: revenue[y] - cac_cost[y] - fixed_cost[y] for y in range(1, 4)}
    npv = sum(profit[y] / (1 + discount_rate)**y for y in range(1, 4)) - initial_investment
    return npv

sensitivity_rows = []
for name, s in strategies.items():
    base_npv = calculate_npv(s, avg_cac)
    for param, mult in [('Конверсия +30%', 1.3), ('Конверсия -30%', 0.7),
                        ('Churn +30%', 1.3), ('Churn -30%', 0.7)]:
        if 'Конверсия' in param:
            npv = calculate_npv(s, avg_cac, conversion_mult=mult)
        else:
            npv = calculate_npv(s, avg_cac, churn_mult=mult)
        sensitivity_rows.append({'Стратегия': name, 'Параметр': param,
                                 'NPV': npv, 'Δ NPV': npv - base_npv})
    # CAC ±30%
    for cac_mult in [1.3, 0.7]:
        npv = calculate_npv(s, avg_cac * cac_mult)
        sensitivity_rows.append({'Стратегия': name, 'Параметр': f'CAC {cac_mult:.0%}',
                                 'NPV': npv, 'Δ NPV': npv - base_npv})

df_sens = pd.DataFrame(sensitivity_rows)
print("\n=== Анализ чувствительности ===")
print(df_sens.to_string(index=False))



=== Анализ чувствительности ===
          Стратегия       Параметр           NPV         Δ NPV
           Freemium Конверсия +30% -7.013207e+07  9.705175e+05
           Freemium Конверсия -30% -7.207310e+07 -9.705175e+05
           Freemium     Churn +30% -7.110259e+07  0.000000e+00
           Freemium     Churn -30% -7.110259e+07  0.000000e+00
           Freemium       CAC 130% -7.161070e+07 -5.081086e+05
           Freemium        CAC 70% -7.059448e+07  5.081086e+05
           Подписка Конверсия +30% -4.279583e+07  7.278881e+06
           Подписка Конверсия -30% -5.735359e+07 -7.278881e+06
           Подписка     Churn +30% -5.007471e+07  0.000000e+00
           Подписка     Churn -30% -5.007471e+07  0.000000e+00
           Подписка       CAC 130% -5.058282e+07 -5.081086e+05
           Подписка        CAC 70% -4.956660e+07  5.081086e+05
       Динамическая Конверсия +30% -5.462401e+07  4.549301e+06
       Динамическая Конверсия -30% -6.372261e+07 -4.549301e+06
       Динамическая   

**Вывод:** наиболее устойчивая стратегия — та, у которой наименьшее абсолютное изменение NPV при варьировании параметров. Обычно это стратегия с высоким LTV/CAC и низким churn — например, **Персонализированная** или **Подписка**.

## Часть 4. Стратегические рекомендации

### Выбор стратегии
На основе расчётов:
- **LTV/CAC > 3** — Подписка и Персонализированная.
- **Наименьшая чувствительность** — Персонализированная (низкий churn 4%).
- **Быстрый рост** — Подписка (высокая конверсия 20%).

**Рекомендация:** комбинированная стратегия — начать с **Подписки** для быстрого набора базы, затем внедрить **Персонализированную** для удержания и монетизации.

### План внедрения
1. **Коммуникационная стратегия:**
   - Контент-маркетинг и SEO (CAC 800 руб.) — основной канал.
   - Виральные механики (реферальные бонусы) — CAC 100 руб.
   - Реклама — только для точечных кампаний.

2. **Дорожная карта:**
   - Месяц 1–3: запуск подписки, тестирование пробного периода.
   - Месяц 4–6: сбор обратной связи, улучшение продукта.
   - Месяц 7–12: внедрение персонализированных тарифов.
   - Год 2–3: масштабирование, выход на 5000+ пользователей.

3. **Ключевые метрики:**
   - LTV/CAC ≥ 3
   - Churn ≤ 5%
   - Конверсия из триала в платных ≥ 20%
   - CAC ≤ 800 руб.
   - NPS ≥ 40

4. **Риски и митигация:**
   - **Высокий churn** → улучшение onboarding, поддержка.
   - **Рост CAC** → развитие виральных каналов.
   - **Конкуренция** → дифференциация за счёт персонализации.
   - **Технические сбои** → резервирование мощностей, SLA.
